# Author capture comparison — JSONL vs JSON

Compare author fields between a `jsonl` and a `json_combined` output of the same anthology. Matches poems by `(title, source_page)` and computes exact match rate and string similarity.

In [ ]:
JSON_PATH  = "pipeline_output/marker_poems/englishpoets02wardiala_poems.json"
JSONL_PATH = "pipeline_output/marker_poems/englishpoets02wardiala_poems.jsonl"

In [ ]:
import json
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd


def _similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.strip().lower(), b.strip().lower()).ratio()


def load_json(path):
    with open(path, encoding="utf-8") as f:
        poems = json.load(f)
    return pd.DataFrame([{
        "title": p.get("title", ""),
        "source_page": p.get("source_page", 0),
        "author_json": p.get("author", ""),
    } for p in poems])


def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = json.loads(line)
            rows.append({
                "title": p.get("title", ""),
                "source_page": p.get("source_page", 0),
                "author_jsonl": p.get("author", ""),
            })
    return pd.DataFrame(rows)


df_json  = load_json(JSON_PATH)
df_jsonl = load_jsonl(JSONL_PATH)

merged = df_json.merge(df_jsonl, on=["title", "source_page"], how="outer", indicator=True)
print(f"JSON: {len(df_json)} poems  |  JSONL: {len(df_jsonl)} poems  |  Matched: {(merged['_merge']=='both').sum()}")

## Poems only in one output

In [ ]:
only_json  = merged[merged["_merge"] == "left_only"][["title", "source_page", "author_json"]]
only_jsonl = merged[merged["_merge"] == "right_only"][["title", "source_page", "author_jsonl"]]

print(f"Only in JSON:  {len(only_json)}")
display(only_json)

print(f"\nOnly in JSONL: {len(only_jsonl)}")
display(only_jsonl)

## Similarity scores

In [ ]:
both = merged[merged["_merge"] == "both"].copy()
both["exact_match"] = both["author_json"] == both["author_jsonl"]
both["similarity"]  = both.apply(lambda r: _similarity(r["author_json"], r["author_jsonl"]), axis=1)

exact_rate = both["exact_match"].mean()
mean_sim   = both["similarity"].mean()

print(f"Matched poems  : {len(both)}")
print(f"Exact match    : {exact_rate:.1%}  ({both['exact_match'].sum()} / {len(both)})")
print(f"Mean similarity: {mean_sim:.3f}")

## Differing authors (sorted worst first)

In [ ]:
changed = (
    both[~both["exact_match"]]
    [["title", "source_page", "author_json", "author_jsonl", "similarity"]]
    .sort_values("similarity")
    .reset_index(drop=True)
)

display(changed)

## Export diff to CSV (optional)

In [ ]:
OUTPUT_CSV = "author_diff.csv"  # set to None to skip

if OUTPUT_CSV:
    both[["title", "source_page", "author_json", "author_jsonl", "exact_match", "similarity"]]\
        .to_csv(OUTPUT_CSV, index=False)
    print(f"Saved to {OUTPUT_CSV}")